# BBC News Text Classification: TF-IDF vs Word2Vec vs DistilBERT vs Zero-Shot LLM

One classification problem, four approaches, benchmarked under matched conditions
(same split, same held-out test set) to compare accuracy, training cost, and data
requirements honestly, rather than assuming a more complex method automatically wins.

**Dataset:** [BBC News Archive](https://www.kaggle.com/datasets/hgultekin/bbcnewsarchive),
2,225 articles across 5 categories (business, entertainment, politics, sport, tech),
originally from Greene & Cunningham (2006). The file is tab-separated, not comma-separated,
noted directly on the dataset page.


## 1. Data loading and split

70/15/15 stratified split by category. No grouping concerns here (unlike the chest X-ray project), each row is an independent article, so a plain stratified split is sufficient.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('/kaggle/input/datasets/hgultekin/bbcnewsarchive/bbc-news-data.csv', sep='\t')

print(f"Total rows: {len(df)}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Missing values:\n{df.isnull().sum()}")
print()
print("Category distribution:")
print(df['category'].value_counts())


In [ ]:
train_df, temp_df = train_test_split(
    df, train_size=0.70, stratify=df['category'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, train_size=0.50, stratify=temp_df['category'], random_state=42
)

for name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f"{name}: {len(split_df)} rows")
    print(split_df['category'].value_counts(normalize=True).round(3))
    print()


Resulting split: 1557 / 334 / 334 (69.9% / 15.0% / 15.0%), with class proportions consistent across all three sets. No rebalancing needed.

## 2. Approach 1: TF-IDF + Logistic Regression

The baseline every other approach is measured against. Unigrams and bigrams, 10,000
max features, standard English stopword removal.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

vectorizer = TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1, 2))

X_train = vectorizer.fit_transform(train_df['content'])
X_val = vectorizer.transform(val_df['content'])
X_test = vectorizer.transform(test_df['content'])

y_train = train_df['category']
y_val = val_df['category']
y_test = test_df['category']

clf_tfidf = LogisticRegression(max_iter=1000, random_state=42)
clf_tfidf.fit(X_train, y_train)

test_preds_tfidf = clf_tfidf.predict(X_test)
test_acc_tfidf = accuracy_score(y_test, test_preds_tfidf)

print(f"Test accuracy (TF-IDF): {test_acc_tfidf:.4f}")
print()
print(classification_report(y_test, test_preds_tfidf))


**Result: 98.80% test accuracy.** A strong baseline. Sport and entertainment near-perfect; politics the weakest category (0.98/0.98), plausibly from vocabulary overlap with business articles on government economic policy.

## 3. Approach 2: Pretrained Word2Vec + Logistic Regression

With only 1,557 training documents, training Word2Vec from scratch would produce weak,
undertrained embeddings, Word2Vec needs a large corpus to learn meaningful word
relationships. Instead, Google's pretrained vectors (300-dim, trained on ~100 billion
words) are used, and each document is represented as the average of its words' vectors.

In [ ]:
import gensim.downloader as api

w2v_model = api.load('word2vec-google-news-300')
print(f"Vocabulary size: {len(w2v_model.index_to_key)}")
print(f"Vector dimension: {w2v_model.vector_size}")


In [ ]:
import numpy as np
import re

def simple_tokenize(text):
    return re.findall(r'\b[a-z]+\b', text.lower())


def document_vector(text, w2v_model):
    """Averages word vectors for a document. Unrecognized words are skipped;
    documents with no recognized words fall back to a zero vector."""
    tokens = simple_tokenize(text)
    vectors = [w2v_model[word] for word in tokens if word in w2v_model]
    if len(vectors) == 0:
        return np.zeros(w2v_model.vector_size)
    return np.mean(vectors, axis=0)


X_train_w2v = np.array([document_vector(t, w2v_model) for t in train_df['content']])
X_val_w2v = np.array([document_vector(t, w2v_model) for t in val_df['content']])
X_test_w2v = np.array([document_vector(t, w2v_model) for t in test_df['content']])

print(f"Train shape: {X_train_w2v.shape}")
zero_vec_count = np.sum(np.all(X_train_w2v == 0, axis=1))
print(f"Documents with zero vector (no recognized words): {zero_vec_count}")


In [ ]:
clf_w2v = LogisticRegression(max_iter=1000, random_state=42)
clf_w2v.fit(X_train_w2v, y_train)

test_preds_w2v = clf_w2v.predict(X_test_w2v)
test_acc_w2v = accuracy_score(y_test, test_preds_w2v)

print(f"Test accuracy (Word2Vec): {test_acc_w2v:.4f}")
print()
print(classification_report(y_test, test_preds_w2v))


**Result: 92.81% test accuracy, clearly below TF-IDF.** A plausible explanation:
averaging word vectors treats every word equally and discards word order and frequency
weighting. This task rewards precise vocabulary matching, a bigram like "interest rate"
or a keyword like "election" is a strong, distinct signal that TF-IDF preserves directly
and that simple averaging tends to wash out. Politics was again the weakest category
(0.88 F1), consistent with the business/politics vocabulary overlap seen in TF-IDF, but
worse here since averaging blurs that distinction further.

## 4. Approach 3: Fine-tuned DistilBERT

Two-phase transfer learning is unnecessary here (unlike the image classifier); the
whole model is fine-tuned directly, standard practice for BERT-family models on
text classification. Checkpointed every epoch and selected by validation loss, not
by whichever epoch finished last, the same discipline used in the image classifier
project, since a preliminary run showed validation loss can rise after epoch 1 on a
dataset this small (early signs of overfitting).

In [ ]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

categories = sorted(train_df['category'].unique())
label2id = {cat: i for i, cat in enumerate(categories)}
id2label = {i: cat for cat, i in label2id.items()}


class NewsDataset(Dataset):
    """Tokenizes article text on the fly. Truncates to 512 tokens (DistilBERT's
    max), which affects a small number of unusually long articles in this dataset,
    a stated limitation rather than an oversight."""
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts.tolist()
        self.labels = [label2id[l] for l in labels]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }


train_dataset_bert = NewsDataset(train_df['content'], train_df['category'], tokenizer)
val_dataset_bert = NewsDataset(val_df['content'], val_df['category'], tokenizer)
test_dataset_bert = NewsDataset(test_df['content'], test_df['category'], tokenizer)

BATCH_SIZE = 8
train_loader_bert = DataLoader(train_dataset_bert, batch_size=BATCH_SIZE, shuffle=True)
val_loader_bert = DataLoader(val_dataset_bert, batch_size=BATCH_SIZE, shuffle=False)
test_loader_bert = DataLoader(test_dataset_bert, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader_bert)}, Val batches: {len(val_loader_bert)}")


In [ ]:
def train_epoch_bert(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate_bert(model, loader, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total


import os
CHECKPOINT_DIR_BERT = '/kaggle/working/bert_checkpoints'
os.makedirs(CHECKPOINT_DIR_BERT, exist_ok=True)

model_bert = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=len(categories),
    id2label=id2label, label2id=label2id
).to(device)

optimizer_bert = AdamW(model_bert.parameters(), lr=2e-5)

NUM_EPOCHS = 3
best_val_loss = float('inf')
best_epoch = None

for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch_bert(model_bert, train_loader_bert, optimizer_bert, device)
    val_loss, val_acc = evaluate_bert(model_bert, val_loader_bert, device)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | train_loss: {train_loss:.4f} | "
          f"val_loss: {val_loss:.4f} | val_acc: {val_acc:.4f}")

    checkpoint_path = os.path.join(CHECKPOINT_DIR_BERT, f'distilbert_epoch{epoch+1}.pt')
    torch.save(model_bert.state_dict(), checkpoint_path)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1

print(f"\nBest epoch: {best_epoch} (val_loss: {best_val_loss:.4f})")


In [ ]:
model_bert.load_state_dict(torch.load(
    os.path.join(CHECKPOINT_DIR_BERT, f'distilbert_epoch{best_epoch}.pt'),
    map_location=device
))

def get_bert_predictions(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels']
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())
    return all_labels, all_preds


test_labels_bert, test_preds_bert = get_bert_predictions(model_bert, test_loader_bert, device)
label_names = [id2label[i] for i in range(len(id2label))]

test_acc_bert = accuracy_score(test_labels_bert, test_preds_bert)
print(f"Test accuracy (DistilBERT): {test_acc_bert:.4f}")
print()
print(classification_report(test_labels_bert, test_preds_bert, target_names=label_names))


**Result: 97.90% test accuracy.** Strong, but did not beat the TF-IDF baseline. A validation-loss check across epochs showed early signs of overfitting are a real risk on a dataset this small; selecting by best validation loss rather than final epoch was necessary, not optional (a preliminary run without checkpointing showed validation loss rising after epoch 1 while training loss kept falling).

## 5. Approach 4: Zero-shot classification via LLM

No training data used at all, the model is prompted with only the category names and
the article text, and asked to classify cold. Uses Groq's API (Llama 3.1 8B Instant)
for fast, low-cost inference across the 334-article test set.

**API key handling:** the key is read from an environment variable, never hardcoded
into the notebook, so it isn't accidentally committed or shared.

In [ ]:
import os
from groq import Groq
import time

# Set this in Kaggle's Secrets/Environment before running, rather than pasting
# the key directly into a cell.
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)

CATEGORIES = ['business', 'entertainment', 'politics', 'sport', 'tech']

def classify_zero_shot(text, client, categories, model="llama-3.1-8b-instant"):
    """Prompts an LLM to classify an article with no training examples, just
    the category names and article text. Truncates long articles to keep the
    prompt within a reasonable size across many calls."""
    truncated_text = text[:3000]

    prompt = f"""Classify the following news article into exactly one of these categories: {', '.join(categories)}.

Respond with ONLY the category name, nothing else.

Article:
{truncated_text}

Category:"""

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=10
    )

    raw_output = response.choices[0].message.content.strip().lower()

    for cat in categories:
        if cat in raw_output:
            return cat
    return None  # unparseable response, tracked separately rather than guessed


In [ ]:
zero_shot_preds, zero_shot_labels = [], []
failed_count = 0

for i in range(len(test_df)):
    text = test_df['content'].iloc[i]
    true_label = test_df['category'].iloc[i]
    pred = classify_zero_shot(text, client, CATEGORIES)

    if pred is None:
        failed_count += 1
        pred = 'business'  # fallback to most common class rather than crashing the loop

    zero_shot_preds.append(pred)
    zero_shot_labels.append(true_label)
    time.sleep(0.1)

print(f"Done. Unparseable responses: {failed_count}")


In [ ]:
zero_shot_acc = accuracy_score(zero_shot_labels, zero_shot_preds)
print(f"Test accuracy (Zero-shot LLM): {zero_shot_acc:.4f}")
print(f"Unparseable responses (defaulted to 'business'): {failed_count}")
print()
print(classification_report(zero_shot_labels, zero_shot_preds))


**Result: 91.02% test accuracy, the lowest of the four.** Business recall dropped
to 0.75 (real business articles missed) while politics precision fell to 0.76 (the
model over-predicted politics broadly), plausibly the same business/politics vocabulary
confusion seen in the other approaches, but sharper here since there is no learned
decision boundary at all, only the model's own general sense of the category names.
One response (of 334) was unparseable and defaulted to the majority class, a real,
stated limitation rather than a hidden one.

## 6. Summary

| Method | Test Accuracy | Training data required | Compute |
|---|---|---|---|
| TF-IDF + Logistic Regression | 98.80% | 1,557 labeled examples | Seconds, CPU |
| DistilBERT (fine-tuned) | 97.90% | 1,557 labeled examples | ~4-5 min, GPU |
| Word2Vec (averaged) + Logistic Regression | 92.81% | 1,557 labeled examples + pretrained vectors | Minutes, CPU |
| Zero-shot LLM (Llama 3.1 8B) | 91.02% | None | API calls, no training |

**Key finding, stated honestly rather than forcing a narrative:** the two approaches that
fit a decision boundary to this specific dataset (TF-IDF, DistilBERT) outperformed the two
that relied on general-purpose representations without task-specific fitting (Word2Vec
averaging, zero-shot). This held regardless of model complexity, the simplest method
(TF-IDF) matched or beat the most complex one (DistilBERT), and BERT fine-tuning provided
no measurable benefit over a simple bag-of-words baseline on this dataset. A plausible
reason: BBC News categories are lexically well-separated (sport vocabulary barely
overlaps with tech vocabulary), a condition under which simpler methods that reward exact
term matching can compete with, or beat, methods built for deeper semantic understanding.

**Practical tradeoffs beyond accuracy:**
- TF-IDF is the cheapest and fastest to train and deploy by a wide margin, no GPU, no
  pretrained downloads, seconds not minutes.
- Zero-shot requires no labeled training data at all, a real advantage in a genuinely
  low-data or rapidly-changing-category situation, despite its lower accuracy here.
- DistilBERT and Word2Vec both add meaningful complexity and compute cost that this
  particular dataset's results don't justify, worth knowing before defaulting to "use
  BERT" as a first instinct on a well-separated classification task.

**Limitations, stated directly:**
- All four methods were evaluated on a single, relatively clean and well-known dataset.
  Results may not generalize to noisier, more ambiguous, or more class-overlapping text.
- DistilBERT fine-tuning shows real overfitting risk on a dataset this small; only
  checkpointing and best-epoch selection avoided reporting a worse-than-optimal result.
- Zero-shot performance depends on prompt wording and the specific LLM used; results are
  not necessarily representative of zero-shot classification in general.
